In [7]:
# Importar librerías necesarias
from flask import Flask, request
import sqlite3

# Crear aplicación Flask
app = Flask(__name__)

# Función para crear la base de datos
def init_db():
    conn = sqlite3.connect('database.db')
    cursor = conn.cursor()

    # Tabla estudiantes
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS estudiantes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT,
        comuna TEXT,
        usuario TEXT,
        password TEXT,
        curso_id INTEGER,
        progreso INTEGER DEFAULT 0
    )
    ''')

    # Tabla cursos
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS cursos (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT,
        tipo TEXT
    )
    ''')

    # Tabla tutores
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS tutores (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        usuario TEXT,
        password TEXT
    )
    ''')

    # Crear tutor por defecto
    cursor.execute("SELECT * FROM tutores")
    if not cursor.fetchall():
        cursor.execute("INSERT INTO tutores (usuario, password) VALUES (?, ?)", ("admin", "1234"))

    conn.commit()
    conn.close()

# Ruta principal (login)
@app.route("/", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        usuario = request.form["usuario"]
        password = request.form["password"]
        tipo = request.form["tipo"]

        conn = sqlite3.connect('database.db')
        cursor = conn.cursor()

        # Validar tutor
        if tipo == "tutor":
            cursor.execute("SELECT * FROM tutores WHERE usuario=? AND password=?", (usuario, password))
            if cursor.fetchone():
                return "<h1>Bienvenido Tutor</h1><a href='/panel_tutor'>Ir al panel</a>"

        # Validar estudiante
        else:
            cursor.execute("SELECT * FROM estudiantes WHERE usuario=? AND password=?", (usuario, password))
            user = cursor.fetchone()
            if user:
                return f"<h1>Bienvenido {user[1]}</h1><a href='/panel_estudiante/{user[0]}'>Ver mi información</a>"

        return "Login incorrecto"

    # 👇 TODO el HTML va aquí adentro
    return """
    <h1>Login</h1>
    <form method="post">
        Usuario: <input name="usuario"><br>
        Password: <input name="password" type="password"><br>
        Tipo:
        <select name="tipo">
            <option value="tutor">Tutor</option>
            <option value="estudiante">Estudiante</option>
        </select><br><br>
        <button>Ingresar</button>
    </form>

    <br><br>

    <a href="/registro_tutor">Registrar como tutor</a>
    """

#AGREGAR REGISTRO DE TUTOR

@app.route("/registro_tutor", methods=["GET", "POST"])
def registro_tutor():
    if request.method == "POST":
        conn = sqlite3.connect('database.db')
        cursor = conn.cursor()

        cursor.execute("""
        INSERT INTO tutores (usuario, password)
        VALUES (?, ?)
        """, (
            request.form["usuario"],
            request.form["password"]
        ))

        conn.commit()
        conn.close()

        return "Tutor registrado <a href='/'>Ir al login</a>"

    return """
    <h1>Registro Tutor</h1>
    <form method="post">
        Usuario: <input name="usuario"><br>
        Password: <input name="password"><br>
        <button>Registrarse</button>
    </form>
    """

#Panel del tutor
@app.route("/panel_tutor")
def panel_tutor():
    return """
    <h1>Panel Tutor</h1>

    <h2>Crear curso</h2>
    <form action="/curso" method="post">
        Nombre: <input name="nombre"><br>
        Tipo:
        <select name="tipo">
            <option>Basica Secundaria</option>
            <option>Tecnico</option>
        </select><br>
        <button>Crear</button>
    </form>

    <h2>Crear estudiante</h2>
    <form action="/estudiante" method="post">
        Nombre: <input name="nombre"><br>
        Comuna: <input name="comuna"><br>
        Usuario: <input name="usuario"><br>
        Password: <input name="password"><br>
        <button>Crear</button>
    </form>

    <h2>Asignar curso</h2>
    <form action="/asignar" method="post">
        ID Estudiante: <input name="estudiante_id"><br>
        ID Curso: <input name="curso_id"><br>
        <button>Asignar</button>
    </form>

    <h2>Registrar progreso</h2>
    <form action="/progreso" method="post">
        ID Estudiante: <input name="estudiante_id"><br>
        Puntos: <input name="puntos"><br>
        <button>Actualizar</button>
    </form>
    """

#FUNCIONES DEL SISTEMA  
#Crear estudiante

@app.route("/estudiante", methods=["POST"])
def estudiante():
    conn = sqlite3.connect('database.db')
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO estudiantes (nombre, comuna, usuario, password)
    VALUES (?, ?, ?, ?)
    """, (
        request.form["nombre"],
        request.form["comuna"],
        request.form["usuario"],
        request.form["password"]
    ))

    conn.commit()
    conn.close()

    return "Estudiante creado <a href='/panel_tutor'>Volver</a>"

#Crear curso
@app.route("/curso", methods=["POST"])
def curso():
    conn = sqlite3.connect('database.db')
    cursor = conn.cursor()

    cursor.execute("INSERT INTO cursos (nombre, tipo) VALUES (?, ?)",
                   (request.form["nombre"], request.form["tipo"]))

    conn.commit()
    conn.close()

    return "Curso creado <a href='/panel_tutor'>Volver</a>"

#Asignar curso
@app.route("/asignar", methods=["POST"])
def asignar():
    conn = sqlite3.connect('database.db')
    cursor = conn.cursor()

    cursor.execute("UPDATE estudiantes SET curso_id=? WHERE id=?",
                   (request.form["curso_id"], request.form["estudiante_id"]))

    conn.commit()
    conn.close()

    return "Curso asignado <a href='/panel_tutor'>Volver</a>"

#Registrar progreso
@app.route("/progreso", methods=["POST"])
def progreso():
    conn = sqlite3.connect('database.db')
    cursor = conn.cursor()

    cursor.execute("UPDATE estudiantes SET progreso = progreso + ? WHERE id=?",
                   (request.form["puntos"], request.form["estudiante_id"]))

    conn.commit()
    conn.close()

    return "Progreso actualizado <a href='/panel_tutor'>Volver</a>"

# PANEL DEL ESTUDIANTE
@app.route("/panel_estudiante/<int:id>")
def panel_estudiante(id):
    conn = sqlite3.connect('database.db')
    cursor = conn.cursor()

    cursor.execute("SELECT nombre, progreso, curso FROM estudiantes WHERE id=?", (id,))
    data = cursor.fetchone()

    conn.close()

    return f"""
    <h1>Bienvenido {data[0]}</h1>
    <p>Curso: {data[2]}</p>
    <p>Progreso: {data[1]} puntos</p>

    <br><br>
    <a href="/cambiar_password/{id}">Cambiar contraseña</a>
    <br>
    <a href="/">Cerrar sesión</a>
    """

#Cambio de constraseña para estudiante

@app.route("/cambiar_password/<int:id>", methods=["GET", "POST"])
def cambiar_password(id):
    if request.method == "POST":
        nueva = request.form["nueva"]

        conn = sqlite3.connect('database.db')
        cursor = conn.cursor()

        cursor.execute("UPDATE estudiantes SET password=? WHERE id=?", (nueva, id))

        conn.commit()
        conn.close()

        return "Contraseña actualizada <a href='/'>Volver al login</a>"

    return """
    <h1>Cambiar contraseña</h1>
    <form method="post">
        Nueva contraseña: <input name="nueva" type="password"><br>
        <button>Actualizar</button>
    </form>
    """
    
def panel_estudiante(id):
    conn = sqlite3.connect('database.db')
    cursor = conn.cursor()

    cursor.execute("""
    SELECT e.nombre, e.progreso, c.nombre
    FROM estudiantes e
    LEFT JOIN cursos c ON e.curso_id = c.id
    WHERE e.id=?
    """, (id,))

    data = cursor.fetchone()
    conn.close()

    return f"""
    <h1>Bienvenido {data[0]}</h1>
    <p>Curso: {data[2]}</p>
    <p>Progreso: {data[1]} puntos</p>
    """


#EJECUTAR LA APP
if __name__ == "__main__":
    app.run(debug=True, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [05/May/2026 13:27:16] "GET /panel_tutor HTTP/1.1" 200 -
127.0.0.1 - - [05/May/2026 13:27:19] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [05/May/2026 13:27:25] "GET /registro_tutor HTTP/1.1" 200 -
127.0.0.1 - - [05/May/2026 13:27:38] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [05/May/2026 13:27:40] "GET /panel_tutor HTTP/1.1" 200 -
